# 9.1 Inspecting Models — Apply

## Objective

Master ONNX model inspection techniques. You will:

1. Build multi-operator models and generate comprehensive summaries
2. Walk through nodes, inputs/outputs, and attributes
3. Build operator histograms and analyze graph topology
4. Compute weight statistics (min, max, mean, std, sparsity)
5. Run and verify model correctness
6. Build a reusable model inspector

**Key ONNX graph structure:**

$$\text{Model} \supset \text{Graph} = (\text{Nodes}, \text{Inputs}, \text{Outputs}, \text{Initializers})$$

Each node computes: $\text{outputs} = \text{op\_type}(\text{inputs}; \text{attributes})$

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
from collections import Counter, defaultdict
import tempfile
import os

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

TMPDIR = tempfile.mkdtemp(prefix='onnx_inspect_')
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")

---
## Exercise 1: Build a Multi-Operator Model

We build a small ConvNet with diverse operator types: Conv, BatchNormalization,
Relu, GlobalAveragePool, Flatten, MatMul, Add, Softmax.

In [ ]:
np.random.seed(42)

def build_demo_model():
    C_in, C1, C2 = 3, 16, 32
    n_classes = 10
    inits = [
        numpy_helper.from_array(np.random.randn(C1, C_in, 3, 3).astype(np.float32)*0.1, 'conv1_W'),
        numpy_helper.from_array(np.zeros(C1, dtype=np.float32), 'conv1_b'),
        numpy_helper.from_array(np.ones(C1, dtype=np.float32), 'bn1_scale'),
        numpy_helper.from_array(np.zeros(C1, dtype=np.float32), 'bn1_bias'),
        numpy_helper.from_array(np.zeros(C1, dtype=np.float32), 'bn1_mean'),
        numpy_helper.from_array(np.ones(C1, dtype=np.float32), 'bn1_var'),
        numpy_helper.from_array(np.random.randn(C2, C1, 3, 3).astype(np.float32)*0.1, 'conv2_W'),
        numpy_helper.from_array(np.zeros(C2, dtype=np.float32), 'conv2_b'),
        numpy_helper.from_array(np.random.randn(C2, n_classes).astype(np.float32)*0.1, 'fc_W'),
        numpy_helper.from_array(np.zeros(n_classes, dtype=np.float32), 'fc_b'),
    ]
    nodes = [
        helper.make_node('Conv', ['X', 'conv1_W', 'conv1_b'], ['conv1'], kernel_shape=[3,3], pads=[1,1,1,1]),
        helper.make_node('BatchNormalization', ['conv1', 'bn1_scale', 'bn1_bias', 'bn1_mean', 'bn1_var'], ['bn1']),
        helper.make_node('Relu', ['bn1'], ['relu1']),
        helper.make_node('Conv', ['relu1', 'conv2_W', 'conv2_b'], ['conv2'], kernel_shape=[3,3], pads=[1,1,1,1]),
        helper.make_node('Relu', ['conv2'], ['relu2']),
        helper.make_node('GlobalAveragePool', ['relu2'], ['gap']),
        helper.make_node('Flatten', ['gap'], ['flat'], axis=1),
        helper.make_node('MatMul', ['flat', 'fc_W'], ['fc']),
        helper.make_node('Add', ['fc', 'fc_b'], ['logits']),
        helper.make_node('Softmax', ['logits'], ['Y'], axis=1),
    ]
    X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 32, 32])
    Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, n_classes])
    g = helper.make_graph(nodes, 'demo_convnet', [X_i], [Y_i], initializer=inits)
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
    m.producer_name = 'onnx_tutorial'
    m.producer_version = '1.0'
    m.doc_string = 'Demo ConvNet for inspection tutorial'
    check_model(m)
    return m

model = build_demo_model()
model_path = os.path.join(TMPDIR, 'demo_model.onnx')
onnx.save(model, model_path)

print(f"Built demo model: {len(model.graph.node)} nodes, {len(model.graph.initializer)} initializers")
print(f"File size: {os.path.getsize(model_path):,} bytes")

---
## Exercise 2: Generate Model Summary (I/O, Metadata, Structure)

A model summary provides the essential information at a glance:
IR version, opset, producer, inputs/outputs with shapes and dtypes.

In [ ]:
def dtype_name(elem_type):
    return TensorProto.DataType.Name(elem_type)

def shape_str(tensor_type):
    dims = []
    for d in tensor_type.shape.dim:
        if d.dim_value:
            dims.append(str(d.dim_value))
        elif d.dim_param:
            dims.append(d.dim_param)
        else:
            dims.append('?')
    return '[' + ', '.join(dims) + ']'

def model_summary(model):
    """Generate a comprehensive model summary."""
    g = model.graph
    init_names = {i.name for i in g.initializer}

    print(f"Model Summary")
    print(f"{'='*55}")
    print(f"  Graph name:      {g.name}")
    print(f"  IR version:      {model.ir_version}")
    print(f"  Producer:        {model.producer_name} {model.producer_version}")
    if model.doc_string:
        print(f"  Doc string:      {model.doc_string}")
    for op in model.opset_import:
        domain = op.domain or 'ai.onnx'
        print(f"  Opset:           {domain} v{op.version}")

    print(f"\n  Inputs:")
    for inp in g.input:
        if inp.name in init_names:
            continue
        tt = inp.type.tensor_type
        print(f"    {inp.name}: {dtype_name(tt.elem_type)} {shape_str(tt)}")

    print(f"  Outputs:")
    for out in g.output:
        tt = out.type.tensor_type
        print(f"    {out.name}: {dtype_name(tt.elem_type)} {shape_str(tt)}")

    print(f"\n  Nodes:          {len(g.node)}")
    print(f"  Initializers:   {len(g.initializer)}")

    total_bytes = sum(numpy_helper.to_array(i).nbytes for i in g.initializer)
    print(f"  Weight bytes:   {total_bytes:,} ({total_bytes/1024:.1f} KB)")

model_summary(model)

---
## Exercise 3: Walk Through Nodes and Print Details

Each ONNX node has:
- `op_type`: the operation (Conv, Relu, MatMul, ...)
- `input`: list of input tensor names
- `output`: list of output tensor names
- `attribute`: operation-specific parameters (kernel_shape, pads, axis, ...)
- `domain`: operator domain (empty = default ai.onnx)
- `name`: optional human-readable name

In [ ]:
def format_attr(attr):
    """Format an ONNX attribute value for display."""
    t = attr.type
    if t == 1: return f"{attr.f}"         # FLOAT
    if t == 2: return f"{attr.i}"         # INT
    if t == 3: return f"'{attr.s.decode()}'"  # STRING
    if t == 6: return f"{list(attr.floats)}"  # FLOATS
    if t == 7: return f"{list(attr.ints)}"    # INTS
    if t == 8: return f"{list(attr.strings)}" # STRINGS
    return f"<type={t}>"

print(f"Detailed Node Walk-Through")
print(f"{'='*70}")
for idx, node in enumerate(model.graph.node):
    domain = node.domain or 'ai.onnx'
    print(f"\n[{idx}] {node.op_type} (domain: {domain})")
    print(f"    inputs:  {list(node.input)}")
    print(f"    outputs: {list(node.output)}")
    if node.attribute:
        print(f"    attributes:")
        for attr in node.attribute:
            print(f"      {attr.name} = {format_attr(attr)}")

# Build an adjacency representation
print(f"\n{'='*70}")
print(f"Graph Connectivity:")
producer = {}  # tensor_name -> node_idx
consumer = defaultdict(list)  # tensor_name -> [node_idx, ...]

for idx, node in enumerate(model.graph.node):
    for out in node.output:
        producer[out] = idx
    for inp in node.input:
        consumer[inp].append(idx)

for idx, node in enumerate(model.graph.node):
    deps = []
    for inp in node.input:
        if inp in producer:
            deps.append(f"node[{producer[inp]}]({model.graph.node[producer[inp]].op_type})")
    dep_str = ' <- ' + ', '.join(deps) if deps else ' <- (graph input/init)'
    print(f"  [{idx}] {node.op_type}{dep_str}")

---
## Exercise 4: Operator Histogram

An operator histogram shows the distribution of operation types.
This is useful for understanding model compute profile and optimization opportunities.

In [ ]:
def op_histogram(model):
    """Compute and display operator histogram."""
    counter = Counter(n.op_type for n in model.graph.node)
    total = len(model.graph.node)

    print(f"Operator Histogram ({total} total nodes)")
    print(f"{'Op Type':<25} {'Count':>5} {'Fraction':>8}  Bar")
    print('-' * 60)
    for op, count in counter.most_common():
        frac = count / total
        bar = '#' * int(frac * 30)
        print(f"{op:<25} {count:>5} {frac:>7.1%}  {bar}")

    # Categorize by compute type
    compute_heavy = {'Conv', 'MatMul', 'Gemm', 'ConvTranspose'}
    normalization = {'BatchNormalization', 'LayerNormalization', 'InstanceNormalization'}
    activation = {'Relu', 'Sigmoid', 'Tanh', 'Softmax', 'LeakyRelu', 'Elu'}
    reshaping = {'Flatten', 'Reshape', 'Squeeze', 'Unsqueeze', 'Transpose'}
    pooling = {'GlobalAveragePool', 'AveragePool', 'MaxPool'}

    categories = {
        'Compute': compute_heavy, 'Normalization': normalization,
        'Activation': activation, 'Reshaping': reshaping,
        'Pooling': pooling,
    }

    print(f"\nBy Category:")
    for cat, ops in categories.items():
        count = sum(counter.get(op, 0) for op in ops)
        if count > 0:
            print(f"  {cat:<15}: {count} ({count/total:.0%})")

    other = total - sum(sum(counter.get(op,0) for op in ops) for ops in categories.values())
    if other > 0:
        print(f"  {'Other':<15}: {other} ({other/total:.0%})")

    return counter

hist = op_histogram(model)

if HAS_MPL:
    ops, counts = zip(*hist.most_common())
    plt.figure(figsize=(8, 4))
    plt.barh(list(reversed(ops)), list(reversed(counts)),
             color='steelblue', edgecolor='black')
    plt.xlabel('Count')
    plt.title('Operator Histogram', fontweight='bold')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Exercise 5: Weight Statistics (Min, Max, Mean, Std)

Weight analysis reveals potential issues: vanishing/exploding weights,
uninitialized parameters, or unexpected distributions.

For a healthy model, weights should have:
- Mean $\approx 0$ (for most layers)
- Std $\approx \sqrt{2/n_{\text{in}}}$ (He initialization)
- No extreme outliers

In [ ]:
def weight_statistics(model):
    """Compute comprehensive weight statistics."""
    stats = []
    for init in model.graph.initializer:
        arr = numpy_helper.to_array(init)
        nnz = np.count_nonzero(arr)
        sparsity = 1.0 - nnz / arr.size if arr.size > 0 else 0.0

        s = {
            'name': init.name,
            'shape': list(arr.shape),
            'dtype': str(arr.dtype),
            'size': arr.size,
            'bytes': arr.nbytes,
            'min': float(arr.min()),
            'max': float(arr.max()),
            'mean': float(arr.mean()),
            'std': float(arr.std()),
            'abs_mean': float(np.abs(arr).mean()),
            'sparsity': sparsity,
            'nnz': nnz,
        }
        stats.append(s)
    return stats

w_stats = weight_statistics(model)

print(f"Weight Statistics")
print(f"{'='*90}")
print(f"{'Name':<15} {'Shape':>15} {'Dtype':>8} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8} {'Sparsity':>9}")
print('-' * 90)
total_bytes = 0
total_params = 0
for s in w_stats:
    shape_str = 'x'.join(str(d) for d in s['shape'])
    print(f"{s['name']:<15} {shape_str:>15} {s['dtype']:>8} "
          f"{s['min']:>8.4f} {s['max']:>8.4f} {s['mean']:>8.4f} {s['std']:>8.4f} {s['sparsity']:>8.1%}")
    total_bytes += s['bytes']
    total_params += s['size']

print(f"\nTotal: {total_params:,} parameters, {total_bytes:,} bytes ({total_bytes/1024:.1f} KB)")

# Check for potential issues
print(f"\nDiagnostics:")
for s in w_stats:
    if s['std'] < 1e-6 and s['size'] > 1:
        print(f"  WARNING: {s['name']} has near-zero std ({s['std']:.2e}) — may be uninitialized")
    if abs(s['mean']) > 1.0:
        print(f"  WARNING: {s['name']} has large mean ({s['mean']:.4f}) — may indicate bias")
    if s['sparsity'] > 0.5:
        print(f"  INFO: {s['name']} is {s['sparsity']:.0%} sparse")

---
## Exercise 6: Graph Topology Analysis

Analyze the graph's structure: depth, width, fan-in/fan-out patterns.

Graph depth (longest path) affects latency:

$$\text{depth} = \max_{\text{path } p \in \text{input}\to\text{output}} |p|$$

Fan-out > 1 indicates tensor reuse (branching). Fan-in > 1 indicates aggregation.

In [ ]:
def graph_topology(model):
    """Analyze graph topology: depth, fan-in, fan-out."""
    g = model.graph
    init_names = {i.name for i in g.initializer}

    # Build producer/consumer maps
    producer = {}
    for idx, node in enumerate(g.node):
        for out in node.output:
            producer[out] = idx

    # Compute depth via longest path from inputs
    depth = {}
    def get_depth(node_idx):
        if node_idx in depth:
            return depth[node_idx]
        node = g.node[node_idx]
        max_parent_depth = 0
        for inp in node.input:
            if inp in producer:
                max_parent_depth = max(max_parent_depth, get_depth(producer[inp]) + 1)
        depth[node_idx] = max_parent_depth
        return max_parent_depth

    for i in range(len(g.node)):
        get_depth(i)

    # Fan-out: how many consumers each tensor has
    tensor_consumers = defaultdict(set)
    for idx, node in enumerate(g.node):
        for inp in node.input:
            tensor_consumers[inp].add(idx)

    max_depth = max(depth.values()) if depth else 0

    print(f"Graph Topology Analysis")
    print(f"{'='*50}")
    print(f"  Total nodes:     {len(g.node)}")
    print(f"  Graph depth:     {max_depth + 1} (longest input-to-output path)")
    print(f"  Unique tensors:  {len(set(t for n in g.node for t in list(n.input)+list(n.output)))}")

    # Depth profile
    depth_counts = Counter(depth.values())
    print(f"\n  Nodes per depth level:")
    for d in sorted(depth_counts.keys()):
        nodes_at_d = [g.node[i].op_type for i, dd in depth.items() if dd == d]
        print(f"    Level {d}: {depth_counts[d]} nodes — {nodes_at_d}")

    # Fan-out analysis
    high_fanout = [(t, len(consumers)) for t, consumers in tensor_consumers.items()
                   if len(consumers) > 1 and t not in init_names]
    if high_fanout:
        print(f"\n  High fan-out tensors (reused by multiple nodes):")
        for t, n in sorted(high_fanout, key=lambda x: -x[1]):
            print(f"    {t}: consumed by {n} nodes")
    else:
        print(f"\n  No branching detected (linear graph)")

    return depth

depth_map = graph_topology(model)

---
## Exercise 7: Run and Verify the Model

After inspection, verify the model works correctly by running it through
ONNX Runtime with random inputs and checking output shapes and value ranges.

In [ ]:
def run_and_verify(model, n_samples=10):
    """Run the model and verify outputs."""
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    inputs = sess.get_inputs()
    outputs = sess.get_outputs()

    print(f"Runtime Verification")
    print(f"{'='*50}")
    print(f"  Provider: {sess.get_providers()}")

    for inp in inputs:
        print(f"  Input: {inp.name}, shape={inp.shape}, type={inp.type}")
    for out in outputs:
        print(f"  Output: {out.name}, shape={out.shape}, type={out.type}")

    # Generate input
    input_info = inputs[0]
    shape = [d if isinstance(d, int) else 1 for d in input_info.shape]

    print(f"\n  Running {n_samples} inferences with shape {shape}...")
    all_outputs = []

    for i in range(n_samples):
        x = np.random.randn(*shape).astype(np.float32)
        result = sess.run(None, {input_info.name: x})
        all_outputs.append(result[0])

    all_outputs = np.array(all_outputs)
    print(f"\n  Output statistics ({n_samples} samples):")
    print(f"    Shape:    {all_outputs.shape[1:]}")
    print(f"    Min:      {all_outputs.min():.6f}")
    print(f"    Max:      {all_outputs.max():.6f}")
    print(f"    Mean:     {all_outputs.mean():.6f}")

    # If Softmax output, verify probabilities sum to 1
    last_op = model.graph.node[-1].op_type
    if last_op == 'Softmax':
        sums = all_outputs.sum(axis=-1)
        print(f"\n  Softmax verification:")
        print(f"    All values in [0,1]: {np.all(all_outputs >= 0) and np.all(all_outputs <= 1)}")
        print(f"    Row sums ≈ 1.0: {np.allclose(sums, 1.0)}")
        print(f"    Min sum: {sums.min():.6f}, Max sum: {sums.max():.6f}")

    # Determinism check
    x_fixed = np.random.randn(*shape).astype(np.float32)
    r1 = sess.run(None, {input_info.name: x_fixed})[0]
    r2 = sess.run(None, {input_info.name: x_fixed})[0]
    print(f"\n  Deterministic: {np.array_equal(r1, r2)}")

run_and_verify(model)

---
## Challenge: Comprehensive Model Inspector

Combine all inspection techniques into a single reusable function that produces
a complete report for any ONNX model.

In [ ]:
def inspect_model(model_or_path):
    """Comprehensive ONNX model inspection."""
    if isinstance(model_or_path, str):
        model = onnx.load(model_or_path)
    else:
        model = model_or_path

    g = model.graph
    init_names = {i.name for i in g.initializer}

    print(f"{'='*60}")
    print(f"  ONNX MODEL INSPECTION REPORT")
    print(f"{'='*60}")

    # Metadata
    print(f"\n[1] METADATA")
    print(f"    Graph: {g.name}")
    print(f"    IR version: {model.ir_version}")
    print(f"    Producer: {model.producer_name} {model.producer_version}")
    for op in model.opset_import:
        print(f"    Opset: {op.domain or 'ai.onnx'} v{op.version}")

    # I/O
    print(f"\n[2] GRAPH I/O")
    for inp in g.input:
        if inp.name in init_names:
            continue
        tt = inp.type.tensor_type
        print(f"    Input:  {inp.name}: {dtype_name(tt.elem_type)} {shape_str(tt)}")
    for out in g.output:
        tt = out.type.tensor_type
        print(f"    Output: {out.name}: {dtype_name(tt.elem_type)} {shape_str(tt)}")

    # Op histogram
    counter = Counter(n.op_type for n in g.node)
    print(f"\n[3] OPERATOR HISTOGRAM ({len(g.node)} nodes)")
    for op, cnt in counter.most_common():
        bar = '#' * min(cnt * 3, 30)
        print(f"    {op:<25} {cnt:>3}  {bar}")

    # Weight summary
    print(f"\n[4] WEIGHT SUMMARY ({len(g.initializer)} initializers)")
    total_params = 0
    total_bytes = 0
    for init in g.initializer:
        arr = numpy_helper.to_array(init)
        total_params += arr.size
        total_bytes += arr.nbytes
        if arr.ndim >= 2:
            shape_s = 'x'.join(str(d) for d in arr.shape)
            print(f"    {init.name:<20} {shape_s:>12} "
                  f"mean={arr.mean():>7.4f} std={arr.std():>7.4f}")
    print(f"    Total: {total_params:,} params, {total_bytes/1024:.1f} KB")

    # Topology
    producer = {}
    for idx, node in enumerate(g.node):
        for out in node.output:
            producer[out] = idx

    depths = {}
    def get_d(ni):
        if ni in depths:
            return depths[ni]
        d = 0
        for inp in g.node[ni].input:
            if inp in producer:
                d = max(d, get_d(producer[inp]) + 1)
        depths[ni] = d
        return d
    for i in range(len(g.node)):
        get_d(i)

    max_d = max(depths.values()) + 1 if depths else 0
    print(f"\n[5] TOPOLOGY")
    print(f"    Graph depth: {max_d}")
    print(f"    Sequential (no branches): {len(set(depths.values())) == len(g.node)}")

    # Validation
    print(f"\n[6] VALIDATION")
    try:
        check_model(model)
        print(f"    check_model: PASS")
    except Exception as e:
        print(f"    check_model: FAIL — {e}")

    try:
        sess = ort.InferenceSession(model.SerializeToString(),
                                    providers=['CPUExecutionProvider'])
        inp = sess.get_inputs()[0]
        shape = [d if isinstance(d, int) else 1 for d in inp.shape]
        x = np.random.randn(*shape).astype(np.float32)
        result = sess.run(None, {inp.name: x})
        print(f"    Inference: PASS (output shape {result[0].shape})")
    except Exception as e:
        print(f"    Inference: FAIL — {e}")

    print(f"\n{'='*60}")

# Run on our demo model
inspect_model(model)

In [ ]:
# Test the inspector on a different model architecture
np.random.seed(0)
dims = [64, 256, 256, 128, 32]
inits, nodes = [], []
prev = 'X'

for i in range(len(dims)-1):
    d_in, d_out = dims[i], dims[i+1]
    inits.append(numpy_helper.from_array(
        np.random.randn(d_in, d_out).astype(np.float32) * np.sqrt(2.0/d_in), f'W{i}'))
    inits.append(numpy_helper.from_array(np.zeros(d_out, dtype=np.float32), f'b{i}'))
    nodes.append(helper.make_node('MatMul', [prev, f'W{i}'], [f'mm{i}']))
    nodes.append(helper.make_node('Add', [f'mm{i}', f'b{i}'], [f'add{i}']))
    if i < len(dims) - 2:
        nodes.append(helper.make_node('Relu', [f'add{i}'], [f'r{i}']))
        prev = f'r{i}'
    else:
        prev = f'add{i}'

X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['N', dims[0]])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['N', dims[-1]])
g = helper.make_graph(nodes, 'deep_mlp', [X_i], [Y_i], initializer=inits)
mlp = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
mlp.producer_name = 'test'

inspect_model(mlp)

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Model summary | IR version, opset, I/O shapes, metadata |
| Node walk-through | op_type, inputs, outputs, attributes for every node |
| Op histogram | Frequency distribution of operations, categorization |
| Weight statistics | Per-initializer min/max/mean/std/sparsity |
| Graph topology | Depth computation, fan-in/fan-out analysis |
| Runtime verification | Inference with random inputs, determinism, Softmax check |
| Model inspector | Reusable comprehensive inspection function |

**Next:** [Modifying Graphs](../02_Modifying_Graphs/)